# T=4.5 by composing two shorter ML propagators

Brand-new notebook; originals are not modified.
K(0,0; 4.5-i0.05) = integral dx K(0,x; 2.25-i0.025)^2 with each half N=8 so a_c matches the N=16 T=4.5 lattice target.


In [1]:
from __future__ import annotations

import csv
import json
import math
import os
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import torch
import torch.nn as nn
from scipy.optimize import root

%matplotlib inline

torch.set_default_dtype(torch.float64)

# Physics: same N=16 complex-time lattice as exact_check_Time_Loop.py
N = 8  # half of the N=16 T=4.5 chain
D = N - 1
ETA = 0.025  # two halves give eta=0.05
MASS = 1.0
OMEGA = 1.0
LAMBDA = 0.5  # held fixed
T_LIST = [2.25]
X_GRID = np.linspace(-3.0, 3.0, 9)
T = float(T_LIST[0])  # current time; updated inside the loop
A_C = (T - 1j * ETA) / N
X_I = X_F = 0.0
EXACT_CSV = Path("Re__Complex_Time_Code_") / "py_scripts_extracted" / "exact_check_Time_Loop_T45_lambda0.5.csv"

# Fastest configuration tested to remain statistically meaningful.
RANK = 16
EPOCHS = 1_000
BATCH = 512
LEARNING_RATE = 1.0e-3
N_MC = 80_000
MC_CHUNK = 5_000
SEED = 52
LOG_EVERY = 250

# Set True only for a syntax check; its numbers are not physics results.
QUICK_TEST = False
if QUICK_TEST:
    EPOCHS, BATCH, N_MC, MC_CHUNK, LOG_EVERY = 5, 32, 1_000, 500, 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
loop_dir = Path("runs") / f"{stamp}_MU_T45_compose_N8x2_lambda0.5"
loop_dir.mkdir(parents=True, exist_ok=True)
run_dir = loop_dir  # overwritten per T inside the loop

print(f"device={DEVICE}; dtype=float64/complex128")
print(f"N={N}, D={D}, lambda={LAMBDA} (fixed), eta={ETA}")
print(f"COMPOSE TEST: two halves T=2.25, N=8, eta=0.025, n_x={len(X_GRID)}")
print(f"rank={RANK}, epochs={EPOCHS} per saddle, batch={BATCH}, N_mc={N_MC}")
print(f"exact lattice CSV: {EXACT_CSV.resolve()}")
print(f"loop directory: {loop_dir.resolve()}")


device=cpu; dtype=float64/complex128
N=8, D=7, lambda=0.5 (fixed), eta=0.025
COMPOSE TEST: two halves T=2.25, N=8, eta=0.025, n_x=9
rank=16, epochs=1000 per saddle, batch=512, N_mc=80000
exact lattice CSV: C:\xampp1\Quantum\Re__Complex_Time_Code_\py_scripts_extracted\exact_check_Time_Loop_T45_lambda0.5.csv
loop directory: C:\xampp1\Quantum\runs\20260927_122414_MU_T45_compose_N8x2_lambda0.5


## 1. Lattice action and stationary points

For zero endpoints, the $D=N-1$ complex interior coordinates have action

$$S[x]=\frac{m}{2a_c}\sum_{k=0}^{N-1}(x_{k+1}-x_k)^2
-\frac{a_cm\omega^2}{2}\sum_{k=1}^{N-1}x_k^2
-\frac{a_c\lambda}{4}\sum_{k=1}^{N-1}x_k^4,$$

with $a_c=(T-i\eta)/N$ and weight $e^{iS}$.  Unlike the harmonic oscillator, the hard
quartic oscillator has nonzero zero-to-zero solutions even when $T<\pi/\omega$: increasing
amplitude raises its frequency.  The previous notebook incorrectly retained only $x=0$.

In [2]:
def action_np(x: np.ndarray) -> complex:
    padded = np.r_[complex(X_I), x, complex(X_F)]
    dx = np.diff(padded)
    vend = 0.5 * (
        0.5 * MASS * OMEGA**2 * (X_I**2 + X_F**2) + 0.25 * LAMBDA * (X_I**4 + X_F**4)
    )
    return (
        MASS / (2.0 * A_C) * np.sum(dx**2)
        - A_C * MASS * OMEGA**2 / 2.0 * np.sum(x**2)
        - A_C * LAMBDA / 4.0 * np.sum(x**4)
        - A_C * vend
    )


def gradient_np(x: np.ndarray) -> np.ndarray:
    padded = np.r_[complex(X_I), x, complex(X_F)]
    return (
        (MASS / A_C) * (2.0 * x - padded[:-2] - padded[2:])
        - A_C * MASS * OMEGA**2 * x
        - A_C * LAMBDA * x**3
    )


def hessian_np(x: np.ndarray) -> np.ndarray:
    hessian = np.zeros((D, D), dtype=np.complex128)
    np.fill_diagonal(
        hessian,
        2.0 * MASS / A_C
        - A_C * MASS * OMEGA**2
        - 3.0 * A_C * LAMBDA * x**2,
    )
    np.fill_diagonal(hessian[1:], -MASS / A_C)
    np.fill_diagonal(hessian[:, 1:], -MASS / A_C)
    return hessian


def packed_gradient(y: np.ndarray) -> np.ndarray:
    x = y[:D] + 1j * y[D:]
    grad = gradient_np(x)
    return np.r_[grad.real, grad.imag]


def action_torch(x: torch.Tensor) -> torch.Tensor:
    x = x.to(torch.complex128)
    xi = torch.as_tensor(complex(X_I), dtype=torch.complex128, device=x.device).expand(x.shape[0], 1).clone()
    xf = torch.as_tensor(complex(X_F), dtype=torch.complex128, device=x.device).expand(x.shape[0], 1).clone()
    padded = torch.cat((xi, x, xf), dim=1)
    dx = padded[:, 1:] - padded[:, :-1]
    a = torch.as_tensor(A_C, dtype=torch.complex128, device=x.device)
    vend = 0.5 * (
        0.5 * MASS * OMEGA**2 * (X_I**2 + X_F**2) + 0.25 * LAMBDA * (X_I**4 + X_F**4)
    )
    return (
        MASS / (2.0 * a) * torch.sum(dx**2, dim=1)
        - a * MASS * OMEGA**2 / 2.0 * torch.sum(x**2, dim=1)
        - a * LAMBDA / 4.0 * torch.sum(x**4, dim=1)
        - a * vend
    )


## 2. Locate the missing bounce and construct local thimble tangents

The first nonzero saddle is found from sine-wave initial guesses and refined against the
full complex lattice equations.  Its parity partner $-x_{\rm b}$ has exactly the same action
and Hessian because the potential is even.

At each saddle, a complex-orthogonal eigendecomposition builds $A_\sigma$ satisfying

$$A_\sigma^T H_\sigma A_\sigma=iI,$$

so $x=x_\sigma+A_\sigma z$ starts on the exact local steepest-descent tangent.  The saddle and
this linear map remain fixed during training, preventing either network from drifting into
the zero saddle or flipping orientation.

In [3]:
def connecting_saddle() -> np.ndarray:
    guess_x = np.linspace(float(X_I), float(X_F), N + 1)[1:-1].astype(np.complex128)
    guess = np.r_[guess_x.real, guess_x.imag]
    solved = root(packed_gradient, guess, method="lm", options={"maxiter": 5_000})
    x = solved.x[:D] + 1j * solved.x[D:]
    residual = np.max(np.abs(gradient_np(x)))
    if residual > 1e-6:
        raise RuntimeError(f"connecting saddle residual {residual:.3e}")
    return x


def first_bounce() -> np.ndarray:
    """Find the lowest-action nonzero zero-endpoint saddle.

    A hard quartic oscillator has amplitude-dependent frequency, so this solution exists
    even though T < pi/omega.  Several amplitudes are tried to avoid accepting x=0.
    """
    sites = np.arange(1, N)
    shape = np.sin(np.pi * sites / N)
    candidates = []
    for amplitude in np.linspace(0.5, 10.0, 20):
        guess = np.r_[amplitude * shape, np.zeros(D)]
        solved = root(packed_gradient, guess, method="lm", options={"maxiter": 5_000})
        x = solved.x[:D] + 1j * solved.x[D:]
        residual = np.max(np.abs(gradient_np(x)))
        if residual < 1e-8 and np.max(np.abs(x)) > 1e-3:
            if not any(np.max(np.abs(x - old)) < 1e-6 for old in candidates):
                candidates.append(x)
    if not candidates:
        raise RuntimeError("Could not find the first nonzero saddle")
    return min(candidates, key=lambda x: abs(action_np(x)))


def tangent_map(hessian: np.ndarray) -> np.ndarray:
    """Return canonically oriented A with A.T @ H @ A = iI."""
    values, vectors = np.linalg.eig(hessian)
    for k in range(D):
        vectors[:, k] /= np.sqrt(vectors[:, k].T @ vectors[:, k])

    # Complex orthogonality implies det(V)=+/-1.  Fix +1 so every saddle uses
    # the same orientation inherited from the real coordinate basis; otherwise
    # a LAPACK eigenvector sign arbitrarily changes K_sigma -> -K_sigma.
    if np.linalg.det(vectors).real < 0.0:
        vectors[:, 0] *= -1.0

    orthogonal_error = np.max(np.abs(vectors.T @ vectors - np.eye(D)))
    orientation_error = abs(np.linalg.det(vectors) - 1.0)
    tangent = vectors @ np.diag(np.sqrt(1j / values))
    residual = np.max(np.abs(tangent.T @ hessian @ tangent - 1j * np.eye(D)))
    if orthogonal_error > 1e-8 or orientation_error > 1e-8 or residual > 1e-8:
        raise RuntimeError(
            "Bad saddle tangent: "
            f"orthogonal={orthogonal_error}, orientation={orientation_error}, "
            f"congruence={residual}"
        )
    return tangent


def fixed_n_reference(basis_size: int = 300) -> complex:
    """Independent deterministic evaluation of this exact N-slice path integral."""
    nb = int(basis_size)
    lowering = np.zeros((nb, nb), dtype=np.float64)
    idx = np.arange(1, nb)
    lowering[idx - 1, idx] = np.sqrt(idx)
    x_op = (lowering + lowering.T) / np.sqrt(2.0)
    p_op = (lowering - lowering.T) / (1j * np.sqrt(2.0))
    x2 = x_op @ x_op

    kinetic = p_op @ p_op / (2.0 * MASS)
    ep, up = scipy.linalg.eigh(kinetic)
    free_step = (up * np.exp(-1j * A_C * ep)) @ up.conj().T

    potential = MASS * OMEGA**2 * x2 / 2.0 + LAMBDA * (x2 @ x2) / 4.0
    ev, uv = scipy.linalg.eigh(potential)
    potential_step = (uv * np.exp(-1j * A_C * ev)) @ uv.conj().T

    position_zero = np.zeros(nb, dtype=np.float64)
    for n in range(0, nb, 2):
        q = n // 2
        position_zero[n] = (-1.0)**q * math.exp(
            0.5 * math.lgamma(2 * q + 1) - q * math.log(2.0)
            - math.lgamma(q + 1) - 0.25 * math.log(np.pi)
        )

    evolution = free_step.copy()
    for _ in range(N - 1):
        evolution = evolution @ potential_step @ free_step
    return complex(position_zero @ evolution @ position_zero)


def gaussian_saddle_estimate(saddle: np.ndarray) -> complex:
    """Quadratic saddle estimate; used only as a decomposition diagnostic."""
    hessian = hessian_np(saddle)
    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    return complex(
        norm * np.exp(1j * action_np(saddle))
        * (2.0 * np.pi * 1j) ** (D / 2) / np.sqrt(np.linalg.det(hessian))
    )

## 3. Saddle-anchored neural contour

For each saddle $x_\sigma$ the learned map is

$$x(z)=x_\sigma+A_\sigma z+C\tanh\!\left(W_2\tanh(W_1z)\right).$$

There are no neural biases, so $g(0)=0$ and the contour always passes through its assigned
saddle.  $A_\sigma$ and $x_\sigma$ are buffers rather than trainable parameters.  Only the
rank-16 nonlinear correction learns.

The full Jacobian is

$$J=A_\sigma+C D_2W_2D_1W_1.$$

For `x = z @ A.T`, Sylvester's identity requires **`solve(A, C)`**.  The old notebook used
`solve(A.T, C)`, i.e. $A^{-T}C$, which is the transpose bug identified in
`Re__Complex_Time_Code_/note_for_tom.pdf`.

In [4]:
class SaddleFlow(nn.Module):
    """Fixed saddle/tangent plus a saddle-anchored low-rank correction."""

    def __init__(self, saddle: np.ndarray, tangent: np.ndarray, seed: int):
        super().__init__()
        torch.manual_seed(seed)
        self.register_buffer(
            "saddle", torch.tensor(saddle, dtype=torch.complex128, device=DEVICE)
        )
        self.register_buffer(
            "A", torch.tensor(tangent, dtype=torch.complex128, device=DEVICE)
        )
        self.W1 = nn.Parameter(torch.randn(RANK, D, device=DEVICE) / math.sqrt(D))
        qmat, _ = torch.linalg.qr(torch.randn(RANK, RANK, device=DEVICE))
        self.W2 = nn.Parameter(qmat)
        # Tiny rather than exactly zero so every layer receives a gradient on step one.
        self.C_real = nn.Parameter(1e-5 * torch.randn(D, RANK, device=DEVICE))
        self.C_imag = nn.Parameter(1e-5 * torch.randn(D, RANK, device=DEVICE))

    def C(self) -> torch.Tensor:
        return self.C_real.to(torch.complex128) + 1j * self.C_imag.to(torch.complex128)

    def forward(self, z: torch.Tensor, return_cache: bool = False):
        z = z.to(dtype=torch.float64, device=DEVICE)
        a1 = torch.tanh(z @ self.W1.T)
        a2 = torch.tanh(a1 @ self.W2.T)
        x = (
            z.to(torch.complex128) @ self.A.T
            + self.saddle
            + a2.to(torch.complex128) @ self.C().T
        )
        return (x, (a1, a2)) if return_cache else x

    def full_jacobian(self, a1: torch.Tensor, a2: torch.Tensor) -> torch.Tensor:
        """Exact batch Jacobian A + C D2 W2 D1 W1, including all cross terms."""
        d1 = 1.0 - a1**2
        d2 = 1.0 - a2**2
        h1 = d1.unsqueeze(-1) * self.W1.unsqueeze(0)
        h2 = torch.matmul(self.W2.unsqueeze(0), h1)
        dh_dz = d2.unsqueeze(-1) * h2
        correction = torch.matmul(self.C().unsqueeze(0), dh_dz.to(torch.complex128))
        return self.A.unsqueeze(0) + correction

    def determinant_parts(self, a1: torch.Tensor, a2: torch.Tensor):
        """Return log|det J| and det(J)/|det(J)| via the corrected low-rank formula."""
        sign_a, logabs_a = torch.linalg.slogdet(self.A)
        # Correct for forward x = z @ A.T.  This is A^{-1}C, not A^{-T}C.
        k_mat = self.W1.to(torch.complex128) @ torch.linalg.solve(self.A, self.C())
        d1 = (1.0 - a1**2).to(torch.complex128)
        d2 = (1.0 - a2**2).to(torch.complex128)
        m1 = d1.unsqueeze(-1) * k_mat.unsqueeze(0)
        m2 = torch.matmul(self.W2.to(torch.complex128).unsqueeze(0), m1)
        correction = (
            torch.eye(RANK, dtype=torch.complex128, device=DEVICE).unsqueeze(0)
            + d2.unsqueeze(-1) * m2
        )
        sign_c, logabs_c = torch.linalg.slogdet(correction)
        return logabs_a + logabs_c, sign_a * sign_c

## 4. Preflight checks

Before training, the notebook verifies:

1. the bounce residual $\|\partial S/\partial x\|_\infty$;
2. parity: $S[x_b]=S[-x_b]$ and $H[x_b]=H[-x_b]$;
3. $A_\sigma^T H_\sigma A_\sigma=iI$;
4. the analytic full Jacobian against central finite differences;
5. the corrected low-rank determinant against `det(full_jacobian)`.

The quadratic saddle estimates provide an independent scale check.  Already at Gaussian
order, subtracting the bounce pair shifts the zero-saddle answer in precisely the direction
required by the deterministic reference.

In [5]:
def validate_saddles_and_jacobian(trivial: np.ndarray, bounce: np.ndarray) -> None:
    residual = np.max(np.abs(gradient_np(bounce)))
    parity_action = abs(action_np(bounce) - action_np(-bounce))
    parity_hessian = np.max(np.abs(hessian_np(bounce) - hessian_np(-bounce)))
    if residual > 1e-8 or parity_action > 1e-10 or parity_hessian > 1e-10:
        raise RuntimeError("Saddle or parity preflight failed")

    tangent = tangent_map(hessian_np(bounce))
    model = SaddleFlow(bounce, tangent, SEED + 999)
    # Make the nonlinear correction large enough that the check tests it, not only A.
    with torch.no_grad():
        model.C_real.normal_(std=0.03)
        model.C_imag.normal_(std=0.03)

    z = torch.randn(3, D, dtype=torch.float64, device=DEVICE)
    x, (a1, a2) = model(z, return_cache=True)
    jacobian = model.full_jacobian(a1, a2)
    eps = 1e-6
    jacobian_fd = torch.empty_like(jacobian)
    with torch.no_grad():
        for k in range(D):
            dz = torch.zeros_like(z)
            dz[:, k] = eps
            jacobian_fd[:, :, k] = (model(z + dz) - model(z - dz)) / (2.0 * eps)
    fd_error = float(torch.max(torch.abs(jacobian - jacobian_fd)).cpu())

    logabs, sign = model.determinant_parts(a1, a2)
    det_low_rank = torch.exp(logabs) * sign
    det_full = torch.linalg.det(jacobian)
    determinant_error = float(torch.max(torch.abs(det_low_rank - det_full)).cpu())
    determinant_relative = determinant_error / max(
        float(torch.max(torch.abs(det_full)).cpu()), 1e-300
    )

    print(f"bounce residual              : {residual:.3e}")
    print(f"parity action/Hessian errors : {parity_action:.3e}, {parity_hessian:.3e}")
    print(f"full Jacobian FD error       : {fd_error:.3e}")
    print(f"low-rank determinant rel err : {determinant_relative:.3e}")
    if fd_error > 3e-7 or determinant_relative > 3e-8:
        raise RuntimeError("Corrected Jacobian preflight failed")
    print("All saddle and Jacobian preflight checks PASSED")

## 5. Branch-safe training and complex importance weights

For standard-normal $z$, write the weight without taking a complex logarithm:

$$\log|W|=-\operatorname{Im}S+\log|\det J|-\log P(z),\qquad
u_W=e^{i\operatorname{Re}S}\frac{\det J}{|\det J|}.$$

The objective is

$$\operatorname{Var}[\log|W|]+1-|\langle\nu_W\rangle|^2.$$

The second term is circular phase variance; unlike `Var(Im(log W))`, it has no artificial
branch cut at $\pm\pi$.  Estimation retains the required
$W=e^{iS}\det J/P$ and the original complex-time normalization.  Five independent MC
blocks expose sampling instability.

In [6]:
LOG_2PI = math.log(2.0 * math.pi)


def weight_parts(model: SaddleFlow, z: torch.Tensor):
    """Branch-safe representation of exp(iS) det(J) / P(z)."""
    x, (a1, a2) = model(z, return_cache=True)
    action = action_torch(x)
    logabs_j, sign_j = model.determinant_parts(a1, a2)
    log_p = -0.5 * torch.sum(z**2, dim=1) - 0.5 * D * LOG_2PI
    logabs_w = -action.imag + logabs_j - log_p
    phase_w = torch.exp(1j * action.real) * sign_j
    return logabs_w, phase_w


def train_thimble(label: str, saddle: np.ndarray, seed: int):
    tangent = tangent_map(hessian_np(saddle))
    model = SaddleFlow(saddle, tangent, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.05
    )
    history = {"total": [], "logmag": [], "phase": []}
    started = time.time()

    for epoch in range(EPOCHS):
        optimizer.zero_grad(set_to_none=True)
        z = torch.randn(BATCH, D, dtype=torch.float64, device=DEVICE)
        logabs_w, phase_w = weight_parts(model, z)
        logmag_variance = torch.var(logabs_w, unbiased=False)
        circular_phase = 1.0 - torch.abs(torch.mean(phase_w))**2
        loss = logmag_variance + circular_phase
        if not torch.isfinite(loss):
            raise FloatingPointError(f"{label}: nonfinite loss at epoch {epoch + 1}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()

        history["total"].append(float(loss.detach().cpu()))
        history["logmag"].append(float(logmag_variance.detach().cpu()))
        history["phase"].append(float(circular_phase.detach().cpu()))
        if epoch == 0 or (epoch + 1) % LOG_EVERY == 0 or epoch + 1 == EPOCHS:
            print(
                f"{label:8s} epoch {epoch+1:4d}/{EPOCHS}: "
                f"loss={history['total'][-1]:.3e}, "
                f"logmag={history['logmag'][-1]:.3e}, "
                f"phase={history['phase'][-1]:.3e}, "
                f"elapsed={time.time()-started:.0f}s",
                flush=True,
            )
    return model, history, time.time() - started


def estimate_thimble(model: SaddleFlow, seed: int):
    """Chunked complex128 MC estimate with five independent stability blocks."""
    torch.manual_seed(seed + 100_000)
    logs, phases = [], []
    remaining = N_MC
    model.eval()
    while remaining:
        count = min(MC_CHUNK, remaining)
        z = torch.randn(count, D, dtype=torch.float64, device=DEVICE)
        with torch.no_grad():
            logabs, phase = weight_parts(model, z)
        logs.append(logabs.cpu())
        phases.append(phase.cpu())
        remaining -= count

    logabs = torch.cat(logs)
    phase = torch.cat(phases)
    shift = torch.max(logabs)
    shifted = torch.exp(logabs - shift) * phase
    norm = (MASS / (2.0 * np.pi * 1j * A_C)) ** (N / 2)
    scale = norm * math.exp(float(shift))
    estimate = complex(scale * torch.mean(shifted).numpy())

    centered = shifted - torch.mean(shifted)
    stderr = abs(scale) * math.sqrt(float(torch.mean(torch.abs(centered)**2)) / N_MC)
    ess = float(
        torch.abs(torch.mean(shifted))**2 / torch.mean(torch.abs(shifted)**2)
    )
    blocks = [
        complex(scale * torch.mean(block).numpy())
        for block in torch.tensor_split(shifted, 5)
    ]
    block_scatter = float(np.std(np.asarray(blocks, dtype=np.complex128)))
    return estimate, stderr, ess, block_scatter, blocks

## 6. Exact lattice targets for $T=0,0.04,\ldots,2$ at $\lambda=0.5$

Load the FFT lattice values produced by `exact_check_Time_Loop.py`.  That is the
correct comparison for this notebook: same $N=16$ discretised path integral.
The continuum column is kept in the table only to show discretisation error.


In [7]:
exact_rows = {}
with EXACT_CSV.open("r", encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        t = round(float(row["T"]), 2)
        exact_rows[t] = {
            "lattice": complex(float(row["lattice_K_re"]), float(row["lattice_K_im"])),
            "continuum": complex(float(row["continuum_K_re"]), float(row["continuum_K_im"])),
        }

print(f"Loaded {len(exact_rows)} exact lattice/continuum rows from {EXACT_CSV}")
print(f"{'T':>6s}  {'lattice K':>32s}  {'continuum K':>32s}")
if 4.5 not in exact_rows:
    raise RuntimeError("T=4.5 missing from full-chain lattice CSV")
print(f"full-chain target T=4.5  lattice={exact_rows[4.5]['lattice']:+.10f}")



def train_half(x_f: float, seed: int):
    global T, A_C, X_I, X_F, run_dir
    T = 2.25
    A_C = (T - 1j * ETA) / N
    X_I = 0.0
    X_F = float(x_f)
    sub = run_dir / f"xf{X_F:+.2f}".replace("+", "p").replace("-", "m")
    sub.mkdir(parents=True, exist_ok=True)
    saddle = connecting_saddle()
    print("\n" + "=" * 84)
    print(f"*** STARTING ML TRAINING: K(0, {X_F:+.2f}; T=2.25)  N=8  (slow step) ***")
    print("=" * 84, flush=True)
    model, history, seconds = train_thimble("connect", saddle, seed)
    estimate, stderr, ess, block_scatter, blocks = estimate_thimble(model, seed)
    print(f"K(0,{X_F:+.2f})={estimate:+.10f}  stderr={stderr:.3e}  ESS/N={ess:.6f}  {seconds:.0f}s")
    np.savez(sub / "saddle.npz", x=saddle, X_F=X_F)
    return {"X_F": X_F, "K": estimate, "stderr": stderr, "ESS": ess, "seconds": seconds}


def run_composition():
    global run_dir
    run_dir = loop_dir / "compose"
    run_dir.mkdir(parents=True, exist_ok=True)
    lattice_ref = exact_rows[4.5]["lattice"]
    samples = []
    for i, xf in enumerate(X_GRID):
        samples.append(train_half(float(xf), SEED + i))
    xs = np.array([s["X_F"] for s in samples], dtype=np.float64)
    ks = np.array([s["K"] for s in samples], dtype=np.complex128)
    esss = np.array([s["ESS"] for s in samples], dtype=np.float64)
    integrand = ks ** 2
    k_comp = np.trapz(integrand, xs)
    err = abs(k_comp - lattice_ref) / abs(lattice_ref)
    print("\n" + "#" * 84)
    print("COMPOSITION RESULT  two x N=8, T=2.25, eta=0.025")
    print("#" * 84)
    for s in samples:
        print(f"x={s['X_F']:+6.2f}  K={s['K']:+.8f}  ESS/N={s['ESS']:.5f}")
    print(f"K_compose  = {k_comp:+.10f}")
    print(f"K_lattice  = {lattice_ref:+.10f}")
    print(f"complex error vs N=16 T=4.5 lattice = {100*err:.3f}%")
    print(f"min ESS/N on the grid = {esss.min():.6f}")
    status = "PASS" if err < 0.05 and esss.min() >= 0.02 else "FAIL"
    row = {
        "T_full": 4.5, "N_half": int(N), "eta_half": ETA,
        "K_compose_re": float(k_comp.real), "K_compose_im": float(k_comp.imag),
        "K_lattice_re": float(lattice_ref.real), "K_lattice_im": float(lattice_ref.imag),
        "complex_rel_err_vs_lattice": float(err),
        "min_ESS_over_N": float(esss.min()),
        "status": status,
        "run_directory": str(run_dir),
    }
    with (run_dir / "results.json").open("w", encoding="utf-8") as handle:
        json.dump(row, handle, indent=2)
    return row


Loaded 1 exact lattice/continuum rows from Re__Complex_Time_Code_\py_scripts_extracted\exact_check_Time_Loop_T45_lambda0.5.csv
     T                         lattice K                       continuum K
full-chain target T=4.5  lattice=-0.9560477521-0.5567694618j


## 7. Train $T=0,0.04,\ldots,2$ at $\lambda=0.5$ and compare to the exact lattice FFT

For each time the notebook rebuilds $a_c=(T-i\eta)/N$, trains the zero thimble and the
first bounce pair, then reports $K_{\rm ML}=K_0+2K_{\rm bounce}$ against the lattice FFT
from `exact_check_Time_Loop.py`.


In [8]:
all_rows = []
comparison_csv = loop_dir / "MU_T45_compose_vs_lattice.csv"
print("*** STARTING ML COMPOSITION NOTEBOOK (9 endpoint trains) ***", flush=True)
row = run_composition()
all_rows.append(row)
with comparison_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
    writer.writeheader()
    writer.writerow(row)
print(f"Wrote {comparison_csv}", flush=True)
print(f"status={row['status']}  err={100*row['complex_rel_err_vs_lattice']:.3f}%")


*** STARTING ML COMPOSITION NOTEBOOK (9 endpoint trains) ***



*** STARTING ML TRAINING: K(0, -3.00; T=2.25)  N=8  (slow step) ***


C:\Users\tomqu\Documents\MLandPython\envs\helios_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


connect  epoch    1/1000: loss=1.378e+00, logmag=1.115e+00, phase=2.635e-01, elapsed=0s


connect  epoch  250/1000: loss=4.125e-01, logmag=2.229e-01, phase=1.897e-01, elapsed=6s


...

connect  epoch  500/1000: loss=2.991e-01, logmag=1.479e-01, phase=1.512e-01, elapsed=13s


connect  epoch  750/1000: loss=3.256e-01, logmag=1.707e-01, phase=1.549e-01, elapsed=19s


connect  epoch 1000/1000: loss=2.587e-01, logmag=1.128e-01, phase=1.459e-01, elapsed=26s


K(0,+3.00)=+0.2626991896-0.2587965746j  stderr=7.953e-03  ESS/N=0.026174  26s

####################################################################################
COMPOSITION RESULT  two x N=8, T=2.25, eta=0.025
####################################################################################
x= -3.00  K=+0.27693261-0.26205842j  ESS/N=0.00930
x= -2.25  K=-0.14285744-0.26954107j  ESS/N=0.00583
x= -1.50  K=-0.05733106-0.04234874j  ESS/N=0.15800
x= -0.75  K=-0.07918291-0.30399155j  ESS/N=0.00624
x= +0.00  K=+0.33770364-0.20552456j  ESS/N=0.97896
x= +0.75  K=-0.08664089-0.29912446j  ESS/N=0.00207
x= +1.50  K=-0.05734358-0.04229173j  ESS/N=0.09382
x= +2.25  K=-0.15523969-0.27470965j  ESS/N=0.57401
x= +3.00  K=+0.26269919-0.25879657j  ESS/N=0.02617
K_compose  = -0.1439229330-0.0055399265j
K_lattice  = -0.9560477521-0.5567694618j
complex error vs N=16 T=4.5 lattice = 88.718%
min ESS/N on the grid = 0.002070
Wrote runs\20260927_122414_MU_T45_compose_N8x2_lambda0.5\MU_T45_compose_vs_lattice

status=FAIL  err=88.718%


## 8. Save the loop CSV

One comparison CSV is written in the loop directory after every $T$, so a
partial run is still usable.  The ML propagator is compared to the **lattice FFT**
column from `exact_check_Time_Loop.py`.


In [9]:
print(f"Saved comparison CSV: {comparison_csv.resolve()}")
print(f"K_compose = {complex(all_rows[0]['K_compose_re'], all_rows[0]['K_compose_im']):+.10f}")
print(f"K_lattice = {complex(all_rows[0]['K_lattice_re'], all_rows[0]['K_lattice_im']):+.10f}")
print(f"err = {100*all_rows[0]['complex_rel_err_vs_lattice']:.3f}%  status={all_rows[0]['status']}")


Saved comparison CSV: C:\xampp1\Quantum\runs\20260927_122414_MU_T45_compose_N8x2_lambda0.5\MU_T45_compose_vs_lattice.csv
K_compose = -0.1439229330-0.0055399265j
K_lattice = -0.9560477521-0.5567694618j
err = 88.718%  status=FAIL


## 9. Summary plot: ML vs exact lattice FFT

Each point is one time at $\lambda=0.5$.  The exact comparison is the lattice FFT, not continuum.


In [10]:
print("composition notebook: single composed T=4.5 point, no T-scan plot")


composition notebook: single composed T=4.5 point, no T-scan plot
